In [8]:
import os, json, ast, re
from pathlib import Path
import numpy as np
from PIL import Image

# ---------------------------
# Text helpers
# ---------------------------
def _normalize_text(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"[\.\,\;\:\!\?\(\)\[\]\{\}\"\']", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s

def split_text(s: str):
    s = _normalize_text(s)
    if not s:
        return []
    return s.split()

def extract_ranked_concepts(token_dict, max_concepts=None):
    """
    token_dict: one entry from per_token_concepts
    returns list[str] ranked by your pipeline similarity
    """
    ranked = token_dict.get("top_concepts", [])
    if max_concepts is not None:
        ranked = ranked[:max_concepts]

    out = []
    for c in ranked:
        tg = c.get("text_grounding", [])
        if isinstance(tg, str):
            # sometimes stored as string repr of list
            try:
                tg = ast.literal_eval(tg)
            except Exception:
                tg = [tg]
        if not tg:
            out.append("")
        else:
            # take first label as representative
            out.append(_normalize_text(tg[0]))
    return out

def lexical_sim(a: str, b: str) -> float:
    """
    light similarity: token overlap ratio.
    Replace with embedding cosine if you want.
    """
    a, b = _normalize_text(a), _normalize_text(b)
    if not a or not b:
        return 0.0
    sa, sb = set(a.split()), set(b.split())
    return len(sa & sb) / max(1, len(sa | sb))

# ---------------------------
# Top-K concept recall
# ---------------------------
def compute_topk_concept_recall(result, ks=(1,2,3), sim_fn=None):
    """
    For each GT token, find best matching concept across ALL tokens,
    then check its rank within that token's ranked list.
    Returns recall@K for each K.
    """
    if sim_fn is None:
        sim_fn = lexical_sim

    gt_tokens = split_text(result.get("model_output", ""))
    per_tokens = result.get("per_token_concepts", [])
    ranked_lists = [extract_ranked_concepts(t) for t in per_tokens]

    best_ranks = []
    for gt in gt_tokens:
        best = (None, None, -1.0)   # (token_idx, rank_idx, score)
        for ti, concepts in enumerate(ranked_lists):
            for ri, ctext in enumerate(concepts):
                s = sim_fn(gt, ctext)
                if s > best[2]:
                    best = (ti, ri, s)
        best_ranks.append(best[1])  # rank index (0-based) or None

    recalls = {}
    for k in ks:
        ok = [(r is not None and r < k) for r in best_ranks]
        recalls[k] = float(np.mean(ok)) if ok else 0.0

    return recalls, best_ranks

# ---------------------------
# BERTScore: top-1 concept text vs prediction tokens
# ---------------------------
def compute_bertscore_concept_vs_pred(result, model_type="microsoft/deberta-xlarge-mnli"):
    """
    Align by token index:
      pred_tokens[i]  vs  top1_concept_text[i]
    Uses bert_score if installed, else falls back to lexical avg.
    """
    pred_tokens = split_text(result.get("model_output", ""))
    per_tokens = result.get("per_token_concepts", [])

    n = max(len(pred_tokens), len(per_tokens)) # Use max as more token genarate per word

    if n == 0:
        return {"P":0.0, "R":0.0, "F1":0.0}

    cands = []
    refs = []
    for i in range(n):
        top1 = extract_ranked_concepts(per_tokens[i], max_concepts=1)[0]
        cands.append(top1 if top1 else "")
        refs.append(pred_tokens[i])

    try:
        from bert_score import score as bert_score
        P, R, F1 = bert_score(cands, refs, lang="en", model_type=model_type, rescale_with_baseline=True)
        return {"P": float(P.mean()), "R": float(R.mean()), "F1": float(F1.mean())}
    except Exception as e:
        # fallback
        sims = [lexical_sim(c, r) for c, r in zip(cands, refs)]
        m = float(np.mean(sims)) if sims else 0.0
        return {"P": m, "R": m, "F1": m, "note": f"bert_score not available ({e}); used lexical fallback"}

# ---------------------------
# CLIPScore: top-1 concept image vs full model output
# ---------------------------
def _get_first_concept_image_path(concept):
    paths = concept.get("image_grounding_path", [])
    if isinstance(paths, str):
        try:
            paths = ast.literal_eval(paths)
        except Exception:
            paths = [paths]
    if not paths:
        return None

    item = paths[0]
    # stored like "idx@/abs/path.png"
    if isinstance(item, str) and "@" in item:
        _, p = item.split("@", 1)
        return p
    return item if isinstance(item, str) else None

def compute_clipscore_conceptimg_vs_output(result, clip_model_name="ViT-B-32"):
    """
    For each token, take top-1 concept's first crop image
    and compute CLIP similarity with full model_output.
    Average over tokens with valid images.
    """
    text = result.get("model_output", "")
    per_tokens = result.get("per_token_concepts", [])

    # lazy-load CLIP
    clip_available = False
    try:
        import open_clip, torch
        model, _, preprocess = open_clip.create_model_and_transforms(clip_model_name, pretrained="openai")
        tokenizer = open_clip.get_tokenizer(clip_model_name)
        model.eval()
        clip_available = True
        backend = "open_clip"
    except Exception:
        try:
            import torch
            from transformers import CLIPProcessor, CLIPModel
            model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
            processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
            model.eval()
            clip_available = True
            backend = "hf_clip"
        except Exception as e:
            return {"clipscore": 0.0, "note": f"CLIP not available ({e})"}

    sims = []
    for tok in per_tokens:
        top_concepts = tok.get("top_concepts", [])
        if not top_concepts:
            continue
        img_path = _get_first_concept_image_path(top_concepts[0])
        if not img_path or not os.path.exists(img_path):
            continue

        img = Image.open(img_path).convert("RGB")

        if backend == "open_clip":
            import torch
            image_in = preprocess(img).unsqueeze(0)
            text_in = tokenizer([text])
            with torch.no_grad():
                im_feat = model.encode_image(image_in)
                tx_feat = model.encode_text(text_in)
                im_feat = im_feat / im_feat.norm(dim=-1, keepdim=True)
                tx_feat = tx_feat / tx_feat.norm(dim=-1, keepdim=True)
                sim = (im_feat * tx_feat).sum(dim=-1).item()
        else:
            import torch
            inputs = processor(text=[text], images=img, return_tensors="pt", padding=True)
            with torch.no_grad():
                outputs = model(**inputs)
                sim = outputs.logits_per_image.item()

        sims.append(sim)

    return {"clipscore": float(np.mean(sims)) if sims else 0.0,
            "n_images": len(sims)}

# ---------------------------
# Run evaluation on a JSON file
# ---------------------------
def evaluate_explanations(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    results = data["results"] if isinstance(data, dict) and "results" in data else data

    all_k1, all_k2, all_k3 = [], [], []
    all_bert_f1 = []
    all_clip = []

    for r in results:
        # 1) Top-K recall
        recalls, _ = compute_topk_concept_recall(r, ks=(1,2,3))
        all_k1.append(recalls[1])
        all_k2.append(recalls[2])
        all_k3.append(recalls[3])

        # 2) BERTScore
        bert = compute_bertscore_concept_vs_pred(r)
        all_bert_f1.append(bert["F1"])

        # 3) CLIPScore
        clip = compute_clipscore_conceptimg_vs_output(r)
        all_clip.append(clip["clipscore"])

    summary = {
        "K@1": float(np.mean(all_k1)) if all_k1 else 0.0,
        "K@2": float(np.mean(all_k2)) if all_k2 else 0.0,
        "K@3": float(np.mean(all_k3)) if all_k3 else 0.0,
        "BERTScore_F1": float(np.mean(all_bert_f1)) if all_bert_f1 else 0.0,
        "CLIPScore": float(np.mean(all_clip)) if all_clip else 0.0,
        "n_samples": len(results)
    }
    return summary


In [9]:
# Configure paths and summarize explanations
ROOT_DIR = Path(os.environ.get("ROOT_DIR", Path.cwd()))
DEFAULT_OUTPUT = ROOT_DIR / "outputs/qwen2_5_10cls_sam/imnet600"
OUTPUT_DIR_BASE = Path(os.environ.get("OUTPUT_DIR", DEFAULT_OUTPUT))
DECOMP_METHOD = os.environ.get("DECOMP_METHOD", "snmf")
EXPLANATIONS_JSON = OUTPUT_DIR_BASE / "explanations" / DECOMP_METHOD / "vlm_explanations.json"

if not EXPLANATIONS_JSON.exists():
    raise FileNotFoundError(f"Missing explanations at {EXPLANATIONS_JSON}")

summary = evaluate_explanations(str(EXPLANATIONS_JSON))
print(json.dumps(summary, indent=2))


KeyboardInterrupt: 